In [1]:
# ============================================================
# Notebook    : 12_paper_assets_export.ipynb
# Description : Extract all publication-ready tables and figures
#               from notebooks 00-11 into a single output folder.
#               English labels/captions throughout (target
#               journals are English-language, per §7).
#
#               Two categories of output:
#                 (A) COPY — already-generated PNGs from SHAP/
#                     fairness notebooks (09/10/11), just
#                     collected and renamed consistently
#                 (B) GENERATE — new tables/figures assembled
#                     from saved model files and prediction
#                     .npz files (never explicitly saved before)
#
#               All outputs go to paper_assets/ with a manifest
#               CSV listing what each file is and which
#               notebook/section it supports.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm pandas numpy matplotlib


# ============================================================
# 1. Common imports and output folder setup
# ============================================================
import os
import shutil
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

np.random.seed(42)
OUT_DIR = "paper_assets"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)

manifest = []  # (filename, type, description, source_notebook)

def log_asset(filename, asset_type, description, source):
    manifest.append({"filename": filename, "type": asset_type,
                      "description": description, "source_notebook": source})
    print(f"  [{asset_type}] {filename}")


# ============================================================
# 2. Helper — safe file check before copy/read
# ============================================================
def check_exists(path, label):
    if os.path.exists(path):
        return True
    print(f"  !! MISSING: {path} ({label}) — skipping, check notebook that should have produced it")
    return False


# ============================================================
# 3. CATEGORY A — copy existing SHAP/fairness figures, renamed
#    with consistent, paper-ready filenames
# ============================================================
print("\n" + "="*60)
print("SECTION A — Copying existing figures (SHAP, fairness)")
print("="*60)

existing_figures = [
    ("data/sequences/shap_case_a_bar.png",       "Fig_SHAP_CaseA_bar.png",
     "Case A (LightGBM-Aggregate) SHAP feature importance (bar)", "09"),
    ("data/sequences/shap_case_a_beeswarm.png",  "Fig_SHAP_CaseA_beeswarm.png",
     "Case A (LightGBM-Aggregate) SHAP summary (beeswarm)", "09"),
    ("data/sequences/shap_case_b_bar.png",       "Fig_SHAP_CaseB_bar.png",
     "Case B (LightGBM-Full) SHAP feature importance (bar)", "09"),
    ("data/sequences/shap_case_b_beeswarm.png",  "Fig_SHAP_CaseB_beeswarm.png",
     "Case B (LightGBM-Full) SHAP summary (beeswarm)", "09"),
    ("data/sequences/shap_case_c_bar.png",       "Fig_SHAP_CaseC_bar.png",
     "Case C (LightGBM-BonusMalus) SHAP feature importance (bar)", "09"),
    ("data/sequences/shap_case_c_beeswarm.png",  "Fig_SHAP_CaseC_beeswarm.png",
     "Case C (LightGBM-BonusMalus) SHAP summary (beeswarm)", "09"),
    ("data/sequences/fairness_case_a_vehtype.png", "Fig_Fairness_CaseA_VehType.png",
     "Case A fairness audit — TPR/FPR by VehType proxy cohort", "10"),
    ("data/sequences/fairness_case_b_tpr_fpr.png", "Fig_Fairness_CaseB_GenderRace.png",
     "Case B fairness audit — TPR/FPR by GENDER and RACE", "11"),
]

for src, dst, desc, notebook in existing_figures:
    dst_path = f"{OUT_DIR}/figures/{dst}"
    if check_exists(src, desc):
        shutil.copy(src, dst_path)
        log_asset(dst, "figure", desc, notebook)


# ============================================================
# 4. CATEGORY A (tables) — copy/relabel existing CSV outputs
# ============================================================
print("\n" + "="*60)
print("SECTION A — Copying existing tables (fairness CSVs)")
print("="*60)

existing_tables = [
    ("data/sequences/fairness_case_a_vehtype_metrics.csv", "Table_Fairness_CaseA_VehType.csv",
     "Case A fairness metrics by VehType (all 16 groups)", "10"),
    ("data/sequences/fairness_case_b_gender_metrics.csv",  "Table_Fairness_CaseB_Gender.csv",
     "Case B fairness metrics by GENDER", "11"),
    ("data/sequences/fairness_case_b_race_metrics.csv",    "Table_Fairness_CaseB_Race.csv",
     "Case B fairness metrics by RACE", "11"),
    ("data/sequences/fairness_case_b_gender_calibration.csv", "Table_Calibration_CaseB_Gender.csv",
     "Case B calibration by GENDER and probability bin", "11"),
    ("data/sequences/fairness_case_b_race_calibration.csv",   "Table_Calibration_CaseB_Race.csv",
     "Case B calibration by RACE and probability bin", "11"),
]

for src, dst, desc, notebook in existing_tables:
    dst_path = f"{OUT_DIR}/tables/{dst}"
    if check_exists(src, desc):
        shutil.copy(src, dst_path)
        log_asset(dst, "table", desc, notebook)


# ============================================================
# 5. CATEGORY B — Table 1: Master results table (10 models,
#    all three Cases). This is §4.4's core table, never saved
#    as a standalone file before — only printed to console
#    across multiple notebooks.
# ============================================================
print("\n" + "="*60)
print("SECTION B — Generating Table 1: Master results (all 10 models)")
print("="*60)

master_results = pd.DataFrame([
    {"Case": "A", "Model_ID": "①",  "Model": "LightGBM-Static",        "Architecture": "LightGBM",    "Feature_Set": "Static only",        "AUC": 0.7686},
    {"Case": "A", "Model_ID": "②",  "Model": "LightGBM-Longitudinal",  "Architecture": "LightGBM",    "Feature_Set": "Static+YearGap",     "AUC": 0.7689},
    {"Case": "A", "Model_ID": "③c", "Model": "LightGBM-Aggregate",     "Architecture": "LightGBM",    "Feature_Set": "Static+YearGap+Agg", "AUC": 0.7919},
    {"Case": "A", "Model_ID": "③",  "Model": "Transformer-Static",     "Architecture": "Transformer", "Feature_Set": "Static only",        "AUC": 0.7664},
    {"Case": "A", "Model_ID": "④",  "Model": "Transformer-Longitudinal","Architecture": "Transformer", "Feature_Set": "Static+YearGap",     "AUC": 0.7669},
    {"Case": "A", "Model_ID": "④b", "Model": "Transformer-Aggregate",  "Architecture": "Transformer", "Feature_Set": "Static+YearGap+Agg", "AUC": 0.7914},
    {"Case": "B", "Model_ID": "B1", "Model": "LightGBM-Demographic",   "Architecture": "LightGBM",    "Feature_Set": "Demographics only",  "AUC": 0.8875},
    {"Case": "B", "Model_ID": "B2", "Model": "LightGBM-Full",          "Architecture": "LightGBM",    "Feature_Set": "Demographics+Behav", "AUC": 0.8894},
    {"Case": "C", "Model_ID": "C1", "Model": "LightGBM-Static",        "Architecture": "LightGBM",    "Feature_Set": "Static only",        "AUC": 0.6788},
    {"Case": "C", "Model_ID": "C2", "Model": "LightGBM-BonusMalus",    "Architecture": "LightGBM",    "Feature_Set": "Static+BonusMalus",  "AUC": 0.7077},
])

master_results.to_csv(f"{OUT_DIR}/tables/Table1_Master_Results.csv", index=False)
log_asset("Table1_Master_Results.csv", "table",
          "Master results table — all 10 models across 3 Cases (AUC)", "03a-08b, assembled")

print(master_results.to_string(index=False))


# ============================================================
# 6. CATEGORY B — Table 2: Case-level improvement summary
#    (§4.4's second table — baseline vs best, improvement, driver)
# ============================================================
print("\n" + "="*60)
print("SECTION B — Generating Table 2: Case-level improvement summary")
print("="*60)

improvement_summary = pd.DataFrame([
    {"Case": "A", "Dataset": "fremotor2freq9907b", "Static_Baseline_AUC": 0.7664, "Static_Baseline_AUC_max": 0.7686,
     "Best_AUC": 0.7919, "Improvement": 0.0919-0.7686+0.7919-0.7919+0.0233,  # placeholder, corrected below
     "Driver": "Aggregate history features (not elapsed time alone)"},
])
# NOTE: compute improvement cleanly instead of the placeholder above
improvement_summary = pd.DataFrame([
    {"Case": "A", "Dataset": "fremotor2freq9907b",
     "Baseline_AUC_range": "0.7664-0.7686", "Best_AUC": 0.7919,
     "Improvement": round(0.7919 - 0.7686, 4),
     "Driver": "Aggregate claim-history features (elapsed time alone contributed <0.001)"},
    {"Case": "B", "Dataset": "car_insurance",
     "Baseline_AUC_range": "0.8875", "Best_AUC": 0.8894,
     "Improvement": round(0.8894 - 0.8875, 4),
     "Driver": "Demographics alone already strong (DRIVING_EXPERIENCE dominant); behavioral history adds little"},
    {"Case": "C", "Dataset": "freMTPL2",
     "Baseline_AUC_range": "0.6788", "Best_AUC": 0.7077,
     "Improvement": round(0.7077 - 0.6788, 4),
     "Driver": "Single compressed history variable (BonusMalus)"},
])

improvement_summary.to_csv(f"{OUT_DIR}/tables/Table2_Case_Improvement_Summary.csv", index=False)
log_asset("Table2_Case_Improvement_Summary.csv", "table",
          "Case-level baseline-to-best improvement and driver (§4.4)", "assembled")
print(improvement_summary.to_string(index=False))


# ============================================================
# 7. CATEGORY B — Figure: Case A 2x3 grid (architecture x
#    feature set), the core visualization for §4.1
# ============================================================
print("\n" + "="*60)
print("SECTION B — Generating Figure: Case A 2x3 AUC grid")
print("="*60)

case_a_grid = master_results[master_results["Case"] == "A"].copy()
pivot = case_a_grid.pivot_table(
    index="Architecture",
    columns="Feature_Set",
    values="AUC"
)
# enforce logical column order (Static -> +YearGap -> +Agg)
pivot = pivot[["Static only", "Static+YearGap", "Static+YearGap+Agg"]]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap="Blues", vmin=0.76, vmax=0.80, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("Case A — AUC by Architecture x Feature Set")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i,j]:.4f}", ha="center", va="center",
                 color="white" if pivot.values[i,j] > 0.785 else "black")
plt.colorbar(im, ax=ax, label="AUC")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/Fig1_CaseA_Architecture_FeatureSet_Grid.png", dpi=150)
plt.close()

log_asset("Fig1_CaseA_Architecture_FeatureSet_Grid.png", "figure",
          "Case A AUC heatmap — Architecture x Feature Set (2x3 grid, §4.1)", "assembled")


# ============================================================
# 8. CATEGORY B — Figure: Cross-Case improvement bar chart
# ============================================================
print("\n" + "="*60)
print("SECTION B — Generating Figure: Cross-Case improvement comparison")
print("="*60)

fig, ax = plt.subplots(figsize=(8, 5))
cases = improvement_summary["Case"]
improvements = improvement_summary["Improvement"]
colors = ["#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar(cases, improvements, color=colors)
ax.set_ylabel("AUC improvement (baseline -> best)")
ax.set_title("Improvement magnitude by Case — what closes the gap differs by data constraint")
for bar, imp, driver in zip(bars, improvements, improvement_summary["Driver"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f"+{imp:.4f}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/Fig2_CrossCase_Improvement.png", dpi=150)
plt.close()

log_asset("Fig2_CrossCase_Improvement.png", "figure",
          "Cross-Case AUC improvement bar chart (§4.4)", "assembled")


# ============================================================
# 9. CATEGORY B — Feature importance tables/figures, regenerated
#    from saved model files (gain-based, LightGBM only — these
#    were only ever printed to console before, never saved)
# ============================================================
print("\n" + "="*60)
print("SECTION B — Regenerating feature importance tables/figures")
print("="*60)

fi_configs = [
    ("data/sequences/lightgbm_aggregate_model.txt",
     ["Expo","YearGap","CumClaimCount","ClaimRateSoFar","YearsSinceFirstClaimLog",
      "PrevLabel","HasPriorClaim","Usage","VehType","VehPower"],
     "CaseA_LightGBM_Aggregate", "Case A (③c LightGBM-Aggregate) feature importance (gain)"),
    ("data/sequences/case_b_lightgbm_full_model.txt",
     ["CREDIT_SCORE","ANNUAL_MILEAGE","CHILDREN","MARRIED","VEHICLE_OWNERSHIP",
      "AGE","GENDER","RACE","DRIVING_EXPERIENCE","EDUCATION","INCOME",
      "VEHICLE_YEAR","VEHICLE_TYPE","CREDIT_SCORE_was_missing",
      "ANNUAL_MILEAGE_was_missing","SPEEDING_VIOLATIONS","DUIS","PAST_ACCIDENTS"],
     "CaseB_LightGBM_Full", "Case B (B2 LightGBM-Full) feature importance (gain)"),
    ("data/sequences/case_c_lightgbm_bonusmalus_model.txt",
     ["Exposure","VehPower","VehAge","DrivAge","Density","BonusMalus",
      "VehBrand","VehGas","Area","Region"],
     "CaseC_LightGBM_BonusMalus", "Case C (C2 LightGBM-BonusMalus) feature importance (gain)"),
]

for model_path, feature_names, tag, desc in fi_configs:
    if not check_exists(model_path, desc):
        continue
    m = lgb.Booster(model_file=model_path)
    gains = m.feature_importance(importance_type="gain")
    fi_df = pd.DataFrame({"feature": feature_names, "gain": gains}).sort_values(
        "gain", ascending=False
    )
    fi_df.to_csv(f"{OUT_DIR}/tables/Table_FeatureImportance_{tag}.csv", index=False)
    log_asset(f"Table_FeatureImportance_{tag}.csv", "table", desc, "regenerated from saved model")

    fig, ax = plt.subplots(figsize=(7, max(3, len(fi_df)*0.3)))
    ax.barh(fi_df["feature"][::-1], fi_df["gain"][::-1], color="#4C72B0")
    ax.set_xlabel("Gain")
    ax.set_title(desc)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/figures/Fig_FeatureImportance_{tag}.png", dpi=150)
    plt.close()
    log_asset(f"Fig_FeatureImportance_{tag}.png", "figure", desc, "regenerated from saved model")


# ============================================================
# 10. Save manifest — a single index of everything in
#     paper_assets/, so nothing has to be re-derived later
# ============================================================
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(f"{OUT_DIR}/MANIFEST.csv", index=False)

print("\n" + "="*60)
print("MANIFEST")
print("="*60)
print(manifest_df.to_string(index=False))


# ============================================================
# 11. Summary
# ============================================================
n_figures = (manifest_df["type"] == "figure").sum()
n_tables  = (manifest_df["type"] == "table").sum()
print("\n===== Paper Assets Export Summary =====")
print(f"Total figures : {n_figures}")
print(f"Total tables  : {n_tables}")
print(f"Output folder : {OUT_DIR}/")
print(f"Manifest      : {OUT_DIR}/MANIFEST.csv")
print("=========================================")


SECTION A — Copying existing figures (SHAP, fairness)
  [figure] Fig_SHAP_CaseA_bar.png
  [figure] Fig_SHAP_CaseA_beeswarm.png
  [figure] Fig_SHAP_CaseB_bar.png
  [figure] Fig_SHAP_CaseB_beeswarm.png
  [figure] Fig_SHAP_CaseC_bar.png
  [figure] Fig_SHAP_CaseC_beeswarm.png
  [figure] Fig_Fairness_CaseA_VehType.png
  [figure] Fig_Fairness_CaseB_GenderRace.png

SECTION A — Copying existing tables (fairness CSVs)
  [table] Table_Fairness_CaseA_VehType.csv
  [table] Table_Fairness_CaseB_Gender.csv
  [table] Table_Fairness_CaseB_Race.csv
  [table] Table_Calibration_CaseB_Gender.csv
  [table] Table_Calibration_CaseB_Race.csv

SECTION B — Generating Table 1: Master results (all 10 models)
  [table] Table1_Master_Results.csv
Case Model_ID                    Model Architecture        Feature_Set    AUC
   A        ①          LightGBM-Static     LightGBM        Static only 0.7686
   A        ②    LightGBM-Longitudinal     LightGBM     Static+YearGap 0.7689
   A       ③c       LightGBM-Aggregate 